# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Raqib-Usman/ML_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/M-Raqib-Usman/ML_Internship.git
%cd ML_Internship

from pathlib import Path
import pandas as pd

RAW = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(RAW)
print(df.shape)
print("Ready.")

Cloning into 'ML_Internship'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 200 (delta 88), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (200/200), 1.91 MiB | 9.30 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/ML_Internship
(30000, 44)
Ready.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

# Paths that work both locally and in Colab after cloning
RAW = Path("data/raw/content_refresh_anonymized.csv")
if not RAW.exists():
    RAW = Path("../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(RAW)

# Official model features from ml_utils.py (these already exclude the leaks)
MODEL_NUMERIC = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

MODEL_CATEGORICAL = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

# Reproduce the exact prep step
for c in ["search_volume","competition","cpc","word_count","char_count",
          "impressions_90d","clicks_90d","pageviews_90d","sessions_90d",
          "users_90d","engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
          "days_with_impressions","days_with_sessions",
          "impressions_last_30d","clicks_last_30d","sessions_last_30d",
          "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d",
          "content_age_days","age_tier_order","days_since_last_update",
          "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct","trend_pct"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

for c in ["competition_level","content_type","main_intent","provider_used","model_used",
          "age_tier","freshness_tier","word_count_tier","char_count_tier",
          "impression_tier","position_tier","trend_direction"]:
    if c in df.columns:
        df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"]     = np.log1p(df["clicks_90d"])
df["log_sessions_90d"]   = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"]= np.log1p(df["ai_sessions_90d"])

X = df[MODEL_NUMERIC + MODEL_CATEGORICAL].copy()
y = df["is_declining_label"]

print("Feature matrix:", X.shape)
print("Base rate (positive class):", round(y.mean(), 3))
print("Features used:", list(X.columns))

Feature matrix: (30000, 26)
Base rate (positive class): 0.542
Features used: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Honest features only
pre = ColumnTransformer([
    ("num", "passthrough", MODEL_NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), MODEL_CATEGORICAL),
])
pipe = Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(max_depth=4, random_state=0))])

honest = cross_val_score(pipe, X, y, cv=3, scoring="roc_auc").mean()

# Deliberately add the leak
X_leaky = X.copy()
X_leaky["trend_pct"] = df["trend_pct"]
leaky = cross_val_score(pipe, X_leaky, y, cv=3, scoring="roc_auc").mean()

print(f"Honest AUC  : {honest:.3f}")
print(f"With leak   : {leaky:.3f}")
print(f"Base rate   : {y.mean():.3f}")
# Expect a large jump → the test can detect leakage.

Honest AUC  : 0.705
With leak   : 0.705
Base rate   : 0.542


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.